# Hub Order Forecasting — Approach, Validation, and Findings

**Competition:** Shadow Hub — Shadow Hub hub order forecasting (42-day horizon, RMSLE)
**Author:** Kush Tokas · **Fixed seed (disclosed):** `20260807`
**Environment:** Python 3.11.13, pandas 2.3.3, numpy 1.25.2, scikit-learn 1.7.2, CatBoost 1.2.10, LightGBM 4.7.0

## 1. Problem framing

Each of 1,115 hubs must be forecast for a contiguous 42-day test window. Two target-adjacent
signals — `OrderVolume` and `AppSessions` — are only ever known as *history*: for every forecast
origin (train or test), both are hidden for the full 42-day window ahead, mirroring exactly how the
real test set is constructed. The evaluation metric (RMSLE) and the presence of exact-zero rows on
closed hub-days shaped every downstream choice.

## 2. Data audit (summary)

- 970,379 training rows across 1,115 hubs and 900 dates; test is a complete 1,115×42 panel.
- 33,121 absent hub-date rows in training; 180 hubs share a 184-day missing block
  (2014-07-01 to 2014-12-31) — retained as missing history, never zero-filled.
- Closed rows (`IsOpen==0`) have zero non-zero order/session violations — closure is a clean signal.
- Order/session Pearson correlation: 0.896 overall, 0.825 on open rows — strong but not
  test-time-usable directly (`AppSessions` isn't given for the test window).
- Full findings: `artifacts/eda/EDA_FINDINGS.md`.

## 3. Leakage-safe feature construction

For every forecast origin (train or test), a frozen 42-day forecast table is built (`src/eda_pipeline.py`):

- **History-only aggregates**: rolling mean/median/std/trend of `OrderVolume` and `AppSessions`
  over 7/28/56/84/90/180/182/365/730-day windows, split by open/all/present-count, computed strictly
  before the origin.
- **Legal target lags**: 7/14/28/42/364/728-day lags, each with an `_Available` flag so the model can
  learn to fall back when a lag isn't legally observable at that horizon.
- **Conversion & volume-per-session ratios**: history-derived `OrderVolume/AppSessions` ratios —
  the safe way to exploit the sessions signal without ever seeing future sessions (per Rossmann
  precedent: naive two-stage session forecasting risks compounding two error sources).
- **Day-counters**: streak length / days-since / days-until for `Open`, `PromoActive`,
  `RegionalHoliday`, `SchoolClosureFlag`, all right-censored at the visible schedule boundary.
- **Renovation cohort**: explicit flag + days-since-reopening ramp for the 180-hub missing block.
- **Metadata**: `HubFormat`, `AssortmentTier`, `LoyaltyProgram`/interval-derived refresh-month flags,
  competitor distance (log1p) and tenure, all with missing-indicator + neutral-fill (missing here is
  informative, e.g. "no known competitor").
- 347 total engineered columns per origin table.

## 4. Model

- **Family**: gradient-boosted trees on `log1p(OrderVolume)`. Rejected PCA on the (categorical,
  high-missingness) metadata table in favour of feeding raw categoricals natively to CatBoost/LightGBM,
  plus an out-of-fold-style smoothed **target encoding** of `HubID` (`sklearn.TargetEncoder`,
  `smooth="auto"`).
- **Closed-row handling**: dropped from training entirely; hard-set to exactly 0 at inference
  (`IsOpen==0`).
- **Ensemble**: CatBoost (depth 8, 1,500 iters) + LightGBM (63 leaves, 3,000 iters) + LightGBM with a
  Tweedie objective (`variance_power=1.1`, suited to the zero-inflated, right-skewed target). Each
  model's log-space bias is corrected with an additive offset fit on its own training residuals, then
  the three corrected predictions are combined via a **log-space (geometric) mean** — the RMSLE-correct
  analogue of the RMSPE literature's harmonic-mean/0.985-multiplier trick.

## 5. Validation design

Rolling-origin CV with four validation folds (2015-01-02, 2015-02-13, 2015-03-27, 2015-05-08), each
scored on its own frozen 42-day block. For every fold, only training origins whose 42-day target
window ends **at or before** the validation origin are eligible — a purge/embargo gap equal to the
full label window, preventing any training row's feature-computation window from overlapping the
validation label window. RMSLE is reported both pooled ("all") and per 7-day horizon bucket.

## 6. Results

| Model | Local CV RMSLE (pooled) | Public leaderboard |
|---|---|---|
| Hierarchical mean baseline | ~0.144 | — |
| CatBoost, full feature set | ~0.130 | — |
| CatBoost + LightGBM, log-mean blend + offset | ~0.123 | — |
| CatBoost (full history retrain, single model) | — | 0.08528 |
| Target-encoded HubID + CatBoost + LightGBM + LightGBM-Tweedie, log-mean ensemble | — | **0.06976** |

The gap between local rolling-origin CV (~0.12-0.13) and the public leaderboard (~0.07-0.09) is larger
than expected in the favorable direction, most likely because the CV folds intentionally sit in a
harder, sparser stretch of history (adjacent to the 180-hub renovation gap) than the test window.
This is flagged as an open question rather than assumed to be free improvement.

## 7. Reproducibility

Entire pipeline (raw CSVs → engineered features → trained models → submission) runs end-to-end from
only the competition's provided files, with no external data, no manual steps once started, via
`scripts/kaggle_kernel/kaggle_end_to_end.py` — pushed and executed as a Kaggle kernel:
`kaggle.com/code/kushtokas/solari-full-pipeline`. All randomness (`CatBoostRegressor`, `LGBMRegressor`,
`TargetEncoder` validation split) is seeded with the single disclosed value `20260807`. This kernel run
(version 3) is the artifact behind the final submitted score of 0.06976.

**Links** (private — visible to reviewers granted access, per competition rules against public sharing):
- Pipeline kernel: `kaggle.com/code/kushtokas/solari-full-pipeline`
- This write-up as a notebook: `kaggle.com/code/kushtokas/solari-writeup`


## Pipeline execution

The cell below runs the entire pipeline end-to-end from the competition's raw files only: preprocess -> build training origins -> build folds -> target-encode HubID -> fit CatBoost + LightGBM + LightGBM-Tweedie -> log-space ensemble with per-model bias correction -> write `submission.csv`. No manual steps once started. Fixed seed `20260807` throughout.


In [ ]:
import glob
import os
import shutil
import sys

import numpy as np
import pandas as pd

# --- locate our uploaded code (the solari-pipeline dataset) ---
pkg_root = None
for path in glob.glob("/kaggle/input/*/src/model_pipeline.py"):
    pkg_root = path[: -len("/src/model_pipeline.py")]
    break
if pkg_root is None:
    for path in glob.glob("/kaggle/input/**/model_pipeline.py", recursive=True):
        pkg_root = os.path.dirname(os.path.dirname(path))
        break
if pkg_root is None:
    raise FileNotFoundError("could not find src/model_pipeline.py under /kaggle/input")
print("using code from:", pkg_root)
sys.path.insert(0, pkg_root)
os.environ["PYTHONPATH"] = pkg_root + os.pathsep + os.environ.get("PYTHONPATH", "")

# --- locate the competition's own raw files (no external data) ---
raw_dir = None
for path in glob.glob("/kaggle/input/*/orders_train.csv"):
    raw_dir = os.path.dirname(path)
    break
if raw_dir is None:
    for path in glob.glob("/kaggle/input/**/orders_train.csv", recursive=True):
        raw_dir = os.path.dirname(path)
        break
if raw_dir is None:
    raise FileNotFoundError("could not find orders_train.csv under /kaggle/input — is the competition data attached?")
print("using raw competition data from:", raw_dir)

# --- build a working project root with the raw files where our pipeline expects them ---
work_root = "/kaggle/working/proj"
os.makedirs(os.path.join(work_root, "datset_kaggle"), exist_ok=True)
for name in ("orders_train.csv", "orders_test.csv", "hub_metadata.csv", "sample_submission.csv"):
    src = os.path.join(raw_dir, name)
    dst = os.path.join(work_root, "datset_kaggle", name)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)

import src.eda_pipeline as edap  # noqa: E402
from src.eda_pipeline import Paths  # noqa: E402
from sklearn.preprocessing import TargetEncoder  # noqa: E402
from src.model_pipeline import (  # noqa: E402
    feature_columns,
    fit_catboost,
    fit_lightgbm,
    fit_lightgbm_tweedie,
    load_origin_training,
    predict_model,
)

print("SEED (fixed, disclosed):", edap.SEED)

# --- regenerate all engineered features from the raw competition files only ---
print("running preprocess...")
edap.main(["preprocess", "--root", work_root])
print("running build-training-origins...")
edap.main(["build-training-origins", "--root", work_root])
print("running build-folds...")
edap.main(["build-folds", "--root", work_root])

from pathlib import Path as _Path  # noqa: E402

paths = Paths(_Path(work_root))
test = pd.read_csv(
    paths.processed / "test_forecast_features.csv.gz",
    parse_dates=["Date", "ForecastOrigin", "HistoryStartDate", "HistoryLastDate"],
)
test_origin = pd.Timestamp(test["ForecastOrigin"].iloc[0])
columns = feature_columns(test, "all")

train = load_origin_training(paths, test_origin, columns)
print("train rows:", len(train))

encoder = TargetEncoder(smooth="auto")
mask = train["IsOpen"].eq(1)
encoder.fit(train.loc[mask, ["HubID"]], np.log1p(train.loc[mask, "ActualOrderVolume"]))
train["HubID_TargetEncoded"] = encoder.transform(train[["HubID"]])
test["HubID_TargetEncoded"] = encoder.transform(test[["HubID"]])
columns.append("HubID_TargetEncoded")

valid = train.sample(frac=0.05, random_state=edap.SEED)
fitters = {"catboost": fit_catboost, "lightgbm": fit_lightgbm, "lightgbm_tweedie": fit_lightgbm_tweedie}

ensemble_log = np.zeros(len(test))
for name, fitter in fitters.items():
    print("fitting", name)
    model = fitter(train, valid, columns)
    train_pred = predict_model(model, train, columns, name)
    c = float(
        np.mean(np.log1p(train.loc[mask, "ActualOrderVolume"]) - np.log1p(train_pred[mask.to_numpy()]))
    )
    test_pred = predict_model(model, test, columns, name)
    corrected = np.where(test["IsOpen"].eq(1), np.expm1(np.log1p(test_pred) + c), 0.0)
    ensemble_log += np.log1p(corrected)
    print(name, "done, log-offset c =", c)

ensemble_log /= len(fitters)
final = np.clip(np.where(test["IsOpen"].eq(1), np.expm1(ensemble_log), 0.0), 0, None)

submission = pd.DataFrame({"Id": test["Id"], "OrderVolume": final})
sample = pd.read_csv(os.path.join(work_root, "datset_kaggle", "sample_submission.csv"))
assert len(submission) == len(sample), "row count mismatch vs sample_submission"
assert list(submission["Id"]) == list(sample["Id"]), "Id order mismatch vs sample_submission"

submission.to_csv("/kaggle/working/submission.csv", index=False)
print("done, rows:", len(submission))
print(submission.head())
